In [ ]:
# download_llama.py
import os
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download, HfApi

# 0) 혹시 켜져있을지 모르는 오프라인 모드 강제 OFF
for v in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"]:
    os.environ.pop(v, None)

# 1) .env 에서 토큰 로드
load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("❌ .env 에 HF_TOKEN 이 없습니다.")

print("HF_TOKEN 앞부분:", hf_token[:10], "...")

# 2) 로그인 + 계정 확인
login(token=hf_token)
who = HfApi().whoami(token=hf_token)
print("✅ 로그인 된 HF 계정:", who["name"])

# 3) 실제 모델 repo_id
REPO_ID = "meta-llama/Llama-3.1-8B-Instruct"
LOCAL_DIR = "./models/llama-3.1-8b-instruct"

print(f"모델 정보: {REPO_ID} → {LOCAL_DIR}")

# 4) 스냅샷 다운로드
path = snapshot_download(
    repo_id=REPO_ID,
    local_dir=LOCAL_DIR,
    local_dir_use_symlinks=False,
    token=hf_token,
)
print("✅ 다운로드 완료, 실제 저장 위치:", path)


C:\Portpolio\yonsei\HrAutoFlow\server\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF_TOKEN 앞부분: hf_ScRDCDx ...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ 로그인 된 HF 계정: yujun-yonsei
모델 정보: meta-llama/Llama-3.1-8B-Instruct → ./models/llama-3.1-8b-instruct


C:\Portpolio\yonsei\HrAutoFlow\server\.venv\Lib\site-packages\huggingface_hub\file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 17 files:  53%|███████████████████████████████████████████████████████████████████████████████████████████                                                                                 | 9/17 [03:05<02:36, 19.54s/it]

In [2]:
import os
import json
from typing import Dict

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
TRAIN_JSONL_PATH = "train.jsonl"
OUTPUT_DIR = "./lora/adapters/resume_scorer_v1"
MAX_SEQ_LENGTH = 2048

# 1) 이 두 줄이 아주 중요: 라이브러리 차원에서 '오프라인 모드' 강제
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

def format_example(example: Dict) -> Dict:
    text = example["input"] + example["output"]
    return {"text": text}

def main():
    # --------------------------
    # 1. 데이터셋 로드
    # --------------------------
    if not os.path.exists(TRAIN_JSONL_PATH):
        raise FileNotFoundError(f"{TRAIN_JSONL_PATH} 를 찾을 수 없습니다.")

    raw_datasets: DatasetDict = load_dataset(
        "json",
        data_files={"train": TRAIN_JSONL_PATH},
    )

    ds = raw_datasets["train"]
    if len(ds) < 2:
        # 샘플이 1개 이하일 때: 그냥 전부 train으로 쓰고 eval은 없음
        train_ds = ds.map(format_example)
        eval_ds = None
    else:
        # 샘플이 2개 이상일 때만 train/test 나누기
        split = ds.train_test_split(test_size=0.1, seed=42)
        train_ds = split["train"]
        eval_ds = split["test"]

    # # train / eval 나누기 (단순 9:1 split)
    # raw_datasets = raw_datasets["train"].train_test_split(test_size=0.1, seed=42)
    # train_ds = raw_datasets["train"].map(format_example)
    # eval_ds = raw_datasets["test"].map(format_example)

    # --------------------------
    # 2. 토크나이저 & 모델 로드 (4bit QLoRA)
    # --------------------------
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"모델 로드 중... ({MODEL_NAME})")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        local_files_only=True,   # 인터넷 절대 안 씀
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        local_files_only=True,   # 인터넷 절대 안 씀
    )

    # --------------------------
    # 3. LoRA 설정
    # --------------------------
    # target_modules 는 모델 구조에 따라 달라질 수 있음 (Llama 계열 기준 예시)
    peft_config = LoraConfig(
        r=64,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    # --------------------------
    # 4. 학습 설정
    # --------------------------
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,  # 효과적인 batch_size ≈ 8
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=200,
        evaluation_strategy="steps",
        eval_steps=200,
        save_total_limit=3,
        bf16=True,  # GPU가 bf16 지원 안 하면 fp16=True 로 바꾸기
        # fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
    )

    # --------------------------
    # 5. SFTTrainer 생성
    # --------------------------
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        peft_config=peft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,  # 여러 샘플을 한 시퀀스로 붙일지 여부 (여기선 False 추천)
        args=training_args,
    )

    # --------------------------
    # 6. 학습 실행
    # --------------------------
    print("학습 시작!")
    trainer.train()

    # --------------------------
    # 7. LoRA 어댑터 + 토크나이저 저장
    # --------------------------
    print("모델 저장 중...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"저장 완료: {OUTPUT_DIR}")
    

if __name__ == "__main__":
    main()


C:\Portpolio\yonsei\HrAutoFlow\server\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 4 examples [00:00, 90.92 examples/s]
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


모델 로드 중... (C:/Users/yujun/.llama/checkpoints/Llama3.1-8B-Instruct)


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory C:/Users/yujun/.llama/checkpoints/Llama3.1-8B-Instruct.